In [21]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [22]:
# Read acceleration data
acceleration_data = pd.read_csv("../output/handshape_acceleration.csv")

acceleration_data.head()

,hand,hand_index,time,acceleration
0,right,0,0.000000,0.022749
1,right,1,0.033333,0.019147
2,right,2,0.066667,0.012561
3,right,3,0.100000,0.004070
4,right,4,0.133333,-0.002786


In [23]:
# Read segment data
segment_data = pd.read_csv("../output/handshape_segments.csv")

segment_data.head()

,hand,start,end
0,left,0,170
1,right,0,36
2,right,41,45
3,right,55,72
4,right,82,170


In [24]:
# # Add hand_index data to acceleration_data (so add an index based on the hand and its)
# acceleration_data['hand_index'] = acceleration_data.groupby('hand').cumcount()

# acceleration_data.head()

In [29]:
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    subplot_titles=("Left Hand Acceleration", "Right Hand Acceleration"),
    vertical_spacing=0.1
)

def get_row(hand_name):
    return 1 if hand_name.lower() == 'left' else 2

for hand in acceleration_data['hand'].unique():  # Per hand (left/right)
    hand_data = acceleration_data[acceleration_data['hand'] == hand]
    row_idx = get_row(hand)

    base_color = '#1f77b4' if hand.lower() == 'left' else '#ff7f0e'

    fig.add_trace(
        go.Scatter(
            x=hand_data['time'],
            y=hand_data['acceleration'],
            mode='lines',
            line=dict(color=base_color, width=1.5),
            name=f"Basis {hand}",
            legendgroup=f"Basis {hand}"
        ),
        row=row_idx, col=1
    )

colours = px.colors.qualitative.Plotly

for index, row in segment_data.iterrows():
    segment_hand = row['hand']
    current_color = colours[index % len(colours)]
    row_idx = get_row(segment_hand)

    segment_slice = acceleration_data[
        (acceleration_data['hand'] == segment_hand) &
        (acceleration_data['hand_index'] >= row['start']) & 
        (acceleration_data['hand_index'] <= row['end'])
    ]

    if not segment_slice.empty:
        fig.add_trace(
            go.Scatter(
                x=segment_slice['time'],
                y=segment_slice['acceleration'],
                mode='lines',
                line=dict(color=current_color, width=4),
                name=f"Seg {index} ({segment_hand})",
                legendgroup=f"Segment {index}",
                showlegend=True
            ),
            row=row_idx, col=1
        )

fig.update_layout(
    title_text="Hand Acceleration Data (per hand)",
    height=600,
    showlegend=True
)

fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="Acceleration (rad/s²)", row=1, col=1)
fig.update_yaxes(title_text="Acceleration (rad/s²)", row=2, col=1)
fig.update_yaxes(matches='y', row=2, col=1)

fig.show()

In [26]:
# Plot acceleration data
fig = px.line(acceleration_data, x='time', y='acceleration', color='hand', title='Hand Acceleration Data')

frame_count = len(acceleration_data['time'].where(acceleration_data['hand'] == 'right').dropna())
t_min = acceleration_data['time'].min()
fps = frame_count / (acceleration_data['time'].max() - t_min)
fps = 30

colours = px.colors.qualitative.Plotly

# 3. Over de segmenten loopen en ze specifiek per hand inkleuren
for index, row in segment_data.iterrows():
    segment_hand = row['hand']
    current_color = colours[index % len(colours)]
    
    # Filter direct op de hand én de hand-specifieke indices (geen tijdsberekening nodig!)
    segment_slice = acceleration_data[
        (acceleration_data['hand'] == segment_hand) &
        (acceleration_data['hand_index'] >= row['start']) & 
        (acceleration_data['hand_index'] <= row['end'])
    ]
    
    # Plot het segment eroverheen
    if not segment_slice.empty:
        fig.add_trace(
            go.Scatter(
                x=segment_slice['time'],
                y=segment_slice['acceleration'],
                mode='lines',
                line=dict(color=current_color, width=4),
                name=f"Seg {index} ({segment_hand})",
                legendgroup=f"Segment {index}",
                showlegend=True
            )
        )

fig.show()